In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# =============================================================================
# BRONZE LAYER — Traffic Volumes Ingestion
# Toronto Chronic Congestion Root-Cause Intelligence Platform
#
# What this notebook does:
#   - Reads the two Traffic Volumes CSV files from the /data folder
#   - Adds an ingestion_timestamp column to track when each record was loaded
#   - Writes both files as raw Bronze Delta tables — NO cleaning, NO filtering
#
# Tables created:
#   - bronze.traffic_volumes_summary   (tmc_most_recent_summary_data)
#   - bronze.traffic_volumes_raw       (tmc_raw_data_2020_2029)
#
# Run this manually once to create the initial tables.
# After that, the Databricks Workflow will call it nightly.
# =============================================================================


# -----------------------------------------------------------------------------
# STEP 1 — Import libraries
# -----------------------------------------------------------------------------
# pyspark.sql.functions gives us the current_timestamp() function
# We use it to stamp every row with the time it was loaded into Bronze

from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# STEP 2 — Define file paths
# -----------------------------------------------------------------------------
# These paths point to where your CSV files live inside your Databricks repo.
# Databricks mounts your GitHub repo at /Workspace/Repos/<your-email>/<repo-name>
# Adjust the email and repo name below to match YOUR Databricks setup.

REPO_BASE = "/Workspace/Repos/panditadvait75@gmail.com/toronto-traffic-pipeline"

SUMMARY_PATH = f"{REPO_BASE}/data/tmc_most_recent_summary_data.csv"
RAW_PATH     = f"{REPO_BASE}/data/tmc_raw_data_2020_2029.csv"


# -----------------------------------------------------------------------------
# STEP 3 — Read the SUMMARY file into a Spark DataFrame
# -----------------------------------------------------------------------------
# inferSchema=True  → Spark automatically detects column types (int, string, etc.)
# header=True       → First row of CSV is treated as column names, not data
# This gives us a DataFrame with all the real columns exactly as they appear in the file

print("Reading summary file...")

df_summary = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SUMMARY_PATH)
)

print(f"Summary file row count: {df_summary.count()}")
print("Summary file columns:")
df_summary.printSchema()


# -----------------------------------------------------------------------------
# STEP 4 — Add ingestion_timestamp to the SUMMARY DataFrame
# -----------------------------------------------------------------------------
# ingestion_timestamp = the exact moment this notebook ran and loaded the data
# This column is a Bronze standard — it tells us WHEN we received this data
# It is NOT in the source file — we add it ourselves
# F.current_timestamp() returns the current date and time as a timestamp type

df_summary = df_summary.withColumn("ingestion_timestamp", F.current_timestamp())


# -----------------------------------------------------------------------------
# STEP 5 — Write SUMMARY DataFrame to Bronze Delta table
# -----------------------------------------------------------------------------
# mode("overwrite") → Replaces the entire table each time this runs
#   Why overwrite for summary?
#   Because this file is a snapshot — it always shows the MOST RECENT count per
#   intersection. It does not accumulate new rows over time like the raw file.
#   So replacing it completely each run is the correct approach.
#
# saveAsTable("bronze.traffic_volumes_summary")
#   → Creates a managed Delta table in the bronze schema
#   → Databricks stores the actual data files in its managed storage
#   → You can then query it with: SELECT * FROM bronze.traffic_volumes_summary

print("Writing summary table to Bronze...")

(
    df_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # Allows schema changes if columns change in future
    .saveAsTable("bronze.traffic_volumes_summary")
)

print("bronze.traffic_volumes_summary written successfully.")


# -----------------------------------------------------------------------------
# STEP 6 — Read the RAW file into a Spark DataFrame
# -----------------------------------------------------------------------------
# The raw file is much larger (83.3MB) — it has one row per 15-minute window
# per intersection per count day. Spark reads it the same way, just more rows.

print("Reading raw file (this may take a moment due to file size)...")

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(RAW_PATH)
)

print(f"Raw file row count: {df_raw.count()}")
print("Raw file columns:")
df_raw.printSchema()


# -----------------------------------------------------------------------------
# STEP 7 — Add ingestion_timestamp to the RAW DataFrame
# -----------------------------------------------------------------------------
# Same pattern as the summary file — stamp every row with the load time

df_raw = df_raw.withColumn("ingestion_timestamp", F.current_timestamp())


# -----------------------------------------------------------------------------
# STEP 8 — Write RAW DataFrame to Bronze Delta table
# -----------------------------------------------------------------------------
# mode("overwrite") → Also overwrites for now
#   Why overwrite for raw too?
#   Because this dataset is a historical snapshot (not a live daily feed).
#   The city published it once and updates it occasionally.
#   When Phase 4 (CDC) runs, we will switch this to "append" + merge logic.
#   For now, a clean overwrite on first load is correct.
#
# partitionBy("count_date")
#   → Physically splits the Delta table files by the count_date column
#   → Makes future queries like WHERE count_date = '2023-06-01' much faster
#   → Spark only reads the relevant partition instead of the entire file
#   → This is important because the raw file is large and has many dates

print("Writing raw table to Bronze (this will take longer due to file size)...")

(
    df_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("count_date")          # Optimize for date-based queries
    .saveAsTable("bronze.traffic_volumes_raw")
)

print("bronze.traffic_volumes_raw written successfully.")


# -----------------------------------------------------------------------------
# STEP 9 — Quick validation checks
# -----------------------------------------------------------------------------
# Sanity check — confirm the tables were created and have data
# These SELECT statements run against the Delta tables we just created

print("\n--- VALIDATION ---")

summary_count = spark.sql("SELECT COUNT(*) as row_count FROM bronze.traffic_volumes_summary").collect()[0]["row_count"]
print(f"bronze.traffic_volumes_summary row count: {summary_count}")

raw_count = spark.sql("SELECT COUNT(*) as row_count FROM bronze.traffic_volumes_raw").collect()[0]["row_count"]
print(f"bronze.traffic_volumes_raw row count: {raw_count}")

# Preview the first 5 rows of each table to confirm structure
print("\nSummary table preview (first 5 rows):")
spark.sql("SELECT * FROM bronze.traffic_volumes_summary LIMIT 5").display()

print("\nRaw table preview (first 5 rows):")
spark.sql("SELECT * FROM bronze.traffic_volumes_raw LIMIT 5").display()

print("\nBronze ingestion complete for Dataset 1 — Traffic Volumes.")